# Пакет 2 (Kaggle-часть): бэкбоны на T4 — Даня ЖУКОВ

Твоя часть матрицы бэкбонов — то, что влезает в одну 12ч сессию T4 (~8-11ч):

| Слот | Модель | Оценка на T4 |
|---|---|---|
| tiny_anchor | cointegrated/rubert-tiny2 | ~1ч |
| rubert_base | ai-forever/ruBert-base | ~6-9ч |
| sbe | fkrasnov2/SBE | ~1ч |

(e5-base и bge-m3 — у Бабкенко на корп-GPU, см. ablation_backbones_babkenko.ipynb.)

Порядок специально: якорь -> base -> sbe. Если сессия/квота кончится на sbe —
главное (base) уже сохранено: результат пишется в json после КАЖДОГО слота.

## Запуск на Kaggle
1. kaggle.com -> Create Notebook -> File -> Import Notebook -> GitHub ->
   `zxcghole228/ecup-2026-product-matching` (ветка egor/llm-solution), этот файл.
2. Add Input -> датасет с данными соревнования + докинь в него файл
   `data_polygon/llm_sample_2m.parquet` из репо.
3. Поправь BASE в конфиге. Settings: GPU T4 x2, Internet On.
4. Save Version -> Save & Run All. Логи смотри на странице версии (бывает лаг).

Всё остальное (bf16/fp16 автоматом, num_workers=0, пропуск упавших, сплит seed 13)
уже учтено в коде — не менять. Якорь должен дать full_macro ~0.70-0.72.


In [ ]:
import os, json, gc, time, random, re
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

# --- ПУТИ: поправь под своё окружение -------------------------------------
DATA_DIR = "/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items"  # <-- поправь
POLYGON_PATH = f"{DATA_DIR}/llm_sample_2m.parquet"
RESULTS_PATH = "/kaggle/working/ablation_results.json"

ITEMS_PATH = f"{DATA_DIR}/items.parquet"
MATCHES_LLM_PATH = f"{DATA_DIR}/matches_llm.parquet"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda"
CC = torch.cuda.get_device_capability(0)
AMP_DTYPE = torch.bfloat16 if CC[0] >= 8 else torch.float16
print(torch.cuda.get_device_name(0), "| amp:", AMP_DTYPE)

# batch подобраны под T4 16GB
EXPERIMENTS = [
    dict(name="tiny_anchor", model="cointegrated/rubert-tiny2", max_len=160, batch=256, accum=1, lr=2e-4),
    dict(name="rubert_base", model="ai-forever/ruBert-base",    max_len=160, batch=96,  accum=3, lr=4e-5),
    dict(name="sbe",         model="fkrasnov2/SBE",             max_len=160, batch=256, accum=1, lr=2e-4),
]

## Данные и общий holdout (seed 13 — НЕ менять)


In [ ]:
ml = pd.read_parquet(MATCHES_LLM_PATH)
parent = {}
def find(x):
    p = parent.setdefault(x, x)
    while p != parent[p]:
        parent[p] = parent[parent[p]]; p = parent[p]
    parent[p] = p; return p
for a, b in zip(ml.id1.values, ml.id2.values):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra
comp = np.fromiter((find(i) for i in ml.id1.values), dtype=np.int64, count=len(ml))
rng = np.random.RandomState(13)
uniq = np.unique(comp)
val_set = set(uniq[rng.rand(len(uniq)) < 0.03].tolist())
is_val = np.fromiter((c in val_set for c in comp), dtype=bool, count=len(ml))
holdout = ml[is_val].copy()
holdout = holdout[(holdout.target <= 0.2) | (holdout.target >= 0.8)]
holdout["target"] = (holdout.target >= 0.5).astype(np.int8)
del ml, parent, comp; gc.collect()

polygon = pd.read_parquet(POLYGON_PATH)
print(f"полигон {len(polygon):,}; holdout {len(holdout):,} (ожидание: 2,170,867 и 191,555)")
assert len(polygon) == 2_170_867, "не тот файл полигона!"

need = set(polygon.id1) | set(polygon.id2) | set(holdout.id1) | set(holdout.id2)
KEY_V1 = ["бренд", "артикул", "партномер", "oem", "код", "модель", "размер",
          "цвет", "объем", "обьем", "вес", "тип", "материал", "количество"]

def build_text_v1(name, attributes):
    parts = [str(name) if name is not None else ""]
    try:
        attrs = json.loads(attributes) if isinstance(attributes, str) else {}
    except Exception:
        attrs = {}
    if isinstance(attrs, dict) and attrs:
        low = {str(k).lower(): str(v) for k, v in attrs.items() if v}
        picked, used = [], set()
        for want in KEY_V1:
            for k, v in low.items():
                if want in k and k not in used:
                    picked.append(f"{k}:{v}"); used.add(k)
        rest = [f"{k}:{v}" for k, v in low.items() if k not in used]
        parts.append(" ; ".join(picked + rest)[:260])
    return " | ".join(parts)

texts, cats = {}, {}
f = pq.ParquetFile(ITEMS_PATH)
for b in f.iter_batches(columns=["id", "name", "attributes", "category"], batch_size=500_000):
    df = b.to_pandas()
    df = df[df["id"].isin(need)]
    for i, n, a, c in df.itertuples(index=False, name=None):
        texts[i] = build_text_v1(n, a); cats[i] = c
holdout["category"] = [cats[i] for i in holdout.id1]
holdout_fast = holdout.sample(min(60_000, len(holdout)), random_state=0)
print(f"товаров: {len(texts):,}")

## Обучение/оценка одного слота


In [ ]:
class DS(Dataset):
    def __init__(self, df):
        self.a = df.id1.values; self.b = df.id2.values
        self.y = df.target.values.astype(np.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return texts[self.a[i]], texts[self.b[i]], self.y[i]

def macro_ap_df(df, preds):
    z = df[["category", "target"]].copy(); z["p"] = preds
    return float(z.groupby("category").apply(
        lambda g: average_precision_score(g.target, g.p)).mean())

@torch.no_grad()
def predict(model, tok, df, max_len, bs=1024):
    model.eval()
    dl = DataLoader(DS(df), batch_size=bs, num_workers=0, shuffle=False,
                    collate_fn=lambda batch: tok([x[0] for x in batch], [x[1] for x in batch],
                        padding=True, truncation=True, max_length=max_len, return_tensors="pt"))
    out = []
    for enc in dl:
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast("cuda", AMP_DTYPE):
            out.append(torch.sigmoid(model(**enc).logits.squeeze(-1).float()).cpu().numpy())
    return np.concatenate(out)

def run_experiment(cfg):
    t_start = time.time()
    tok = AutoTokenizer.from_pretrained(cfg["model"])
    model = AutoModelForSequenceClassification.from_pretrained(cfg["model"], num_labels=1).to(device)
    max_len = cfg["max_len"]
    limit = getattr(tok, "model_max_length", 10**9)
    if limit and limit < max_len:
        print(f"  ! {cfg['model']}: model_max_length={limit} < {max_len} — зажимаю")
        max_len = int(limit)
    def collate(batch):
        enc = tok([x[0] for x in batch], [x[1] for x in batch], padding=True,
                  truncation=True, max_length=max_len, return_tensors="pt")
        return enc, torch.tensor([x[2] for x in batch])
    dl = DataLoader(DS(polygon), batch_size=cfg["batch"], shuffle=True,
                    num_workers=0, drop_last=True, collate_fn=collate)
    steps = len(dl) // cfg["accum"]
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, steps // 30, steps)
    use_scaler = AMP_DTYPE == torch.float16
    scaler = torch.amp.GradScaler(enabled=use_scaler)
    lossf = nn.BCEWithLogitsLoss()
    model.train(); t0 = time.time(); seen = 0
    for bi, (enc, y) in enumerate(dl):
        enc = {k: v.to(device, non_blocking=True) for k, v in enc.items()}
        y = y.to(device, non_blocking=True)
        with torch.autocast("cuda", AMP_DTYPE):
            loss = lossf(model(**enc).logits.squeeze(-1), y) / cfg["accum"]
        scaler.scale(loss).backward() if use_scaler else loss.backward()
        if (bi + 1) % cfg["accum"] == 0:
            if use_scaler:
                scaler.step(opt); scaler.update()
            else:
                opt.step()
            opt.zero_grad(set_to_none=True); sched.step()
        seen += len(y)
        if seen % (cfg["batch"] * cfg["accum"] * 400) < cfg["batch"]:
            print(f"  [{cfg['name']}] {seen:,}/{len(polygon):,} {seen/(time.time()-t0):.0f} pair/s", flush=True)
    fast = macro_ap_df(holdout_fast, predict(model, tok, holdout_fast, max_len))
    full = macro_ap_df(holdout, predict(model, tok, holdout, max_len))
    res = dict(cfg, used_max_len=max_len, fast_macro=round(fast, 4),
               full_macro=round(full, 4), minutes=round((time.time()-t_start)/60, 1))
    del model; gc.collect(); torch.cuda.empty_cache()
    return res

In [ ]:
results = []
if os.path.exists(RESULTS_PATH):
    results = json.load(open(RESULTS_PATH))
    print("уже готово:", [r["name"] for r in results])
done = {r["name"] for r in results}
for cfg in EXPERIMENTS:
    if cfg["name"] in done:
        continue
    print(f"=== {cfg['name']}: {cfg['model']} ===", flush=True)
    try:
        res = run_experiment(cfg)
    except Exception as e:
        print(f"  !! {cfg['name']}: {type(e).__name__}: {e} — пропускаю", flush=True)
        gc.collect(); torch.cuda.empty_cache()
        continue
    results.append(res)
    json.dump(results, open(RESULTS_PATH, "w"), ensure_ascii=False, indent=1)
    print(pd.DataFrame(results)[["name","model","used_max_len","fast_macro","full_macro","minutes"]]
          .to_string(index=False), flush=True)
print("ГОТОВО")

## Как читать и что вернуть команде

- Якорь tiny_anchor должен дать ~0.70-0.72 full_macro. Сильно меньше -> пути/данные битые, остальному не верить.
- Победитель = максимальный full_macro среди base+ моделей. Он едет на полные 11.2M (2 эпохи).
- SBE выше tiny при вдвое меньшем размере -> домен важнее размера, обсуждаем поиск доменных моделей крупнее.
- Вернуть: ablation_results.json + строку в таблицу solution/README.md + исполненный ноутбук в notebooks/runs/.
- Метрики полигона ниже полных на ~0.02-0.04 — важны разницы, не абсолюты.
